In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lit, row_number, countDistinct, count, sum, avg, max, min, mean, stddev, skewness, kurtosis, corr, covar_pop, covar_samp, collect_list, collect_set, array_contains, array_join, array_sort, array_remove, array_distinct, array_union, array_intersect, array_except, arrays_zip, array_repeat, array_contains, array_position, array_sort, array_remove, array_distinct, array_union, array_intersect, array_except, arrays_zip, array_repeat, array_contains

# Data Reading

In [0]:
df = spark.read.format("parquet")\
                .load("abfss://bronze-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/products")

In [0]:
display(df)

In [0]:
df = df.drop("_rescued_data")

display(df)

# Create View using pyspark

In [0]:
df.createOrReplaceTempView("products")

# Functions in Unity Catalog

In [0]:
%sql
CREATE OR REPLACE FUNCTION medallion_arc_e2e.bronze.discount(p_price DOUBLE)
RETURNS DOUBLE
language SQL
RETURN p_price * 0.90

In [0]:
%sql
SELECT product_id, medallion_arc_e2e.bronze.discount(price) as discounted_price 
FROM products

In [0]:
df = df.withColumn("discounted_price", expr("medallion_arc_e2e.bronze.discount(price)"))

display(df)

# Function Using Python Or Pyspark

In [0]:
%sql
CREATE OR REPLACE FUNCTION medallion_arc_e2e.bronze.uppercase(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$

  return p_brand.upper()

$$

In [0]:
%sql
SELECT product_id, brand, medallion_arc_e2e.bronze.uppercase(brand) AS brand_upper
FROM products

In [0]:
df.write.format("delta").mode("append")\
                        .option("path","abfss://silver-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/products")\
                        .saveAsTable("medallion_arc_e2e.silver.products")

In [0]:
# Register as external table in Unity Catalog
spark.sql("""
CREATE TABLE IF NOT EXISTS medallion_arc_e2e.silver.products
USING DELTA
LOCATION 'abfss://silver-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/products'
""")

In [0]:
try:    df = spark.read.format("delta").load("abfss://silver-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/products")
except AnalysisException as e:
    print(f"Error loading data: {e}")


display(df)